# Clustering related findings
### Author: Oscar Heath
### Date: Oct 12th 2026

Takes the pairs judged related in `link_findings.ipynb` (`hypotheses_related_findings.parquet`), treats each related pair as an edge between two findings, and clusters the resulting graph with Leiden. Every finding in `hypotheses_processed_results.parquet` is a node, so findings with no related pairs end up in their own singleton cluster.

In [ ]:
import leidenalg as la
import igraph as ig
import polars as pl

In [ ]:
findings = pl.read_parquet("../../data/hypotheses_processed_results.parquet")
related = pl.read_parquet("../../data/hypotheses_related_findings.parquet")

# row_1 and row_2 index into the processed results, the graph is undirected so (a, b) and (b, a) are the same edge
edges = (
    related.filter(pl.col("row_1") != pl.col("row_2"))
    .select(
        pl.min_horizontal("row_1", "row_2").alias("source"),
        pl.max_horizontal("row_1", "row_2").alias("target"),
    )
    .unique()
)
print(f"{edges.height} unique edges between {findings.height} findings")

In [ ]:
g = ig.Graph(n=findings.height, edges=edges.rows(), directed=False)

# CPM finds clusters whose internal edge density is above the resolution parameter,
# higher values give smaller, tighter clusters
RESOLUTION = 0.2

partition = la.find_partition(
    g,
    la.CPMVertexPartition,
    resolution_parameter=RESOLUTION,
    n_iterations=-1,
    seed=42,
)
print(f"{len(partition)} clusters, quality {partition.quality():.2f}")

In [ ]:
# partition.membership gives the cluster id of each vertex, and vertex i is row i of findings
clustered = findings.with_columns(pl.Series("cluster_id", partition.membership))

cluster_sizes = clustered.group_by("cluster_id").len().sort("len", descending=True)
print(f"{cluster_sizes.filter(pl.col('len') > 1).height} clusters with more than one finding")
cluster_sizes.head(20)

In [ ]:
clustered.write_parquet("../../data/hypotheses_clusters.parquet")